In [18]:
# Оценка качества моделей
# MSE, R², AIC, BIC, анализ остатков (нормальность, АКФ, гомоскедастичность)

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

In [20]:
import os
os.chdir("/Users/maksimtimosuk/Desktop/models")

In [21]:
# Загрузка сохранённых результатов моделей

In [22]:
arima_params = np.load("arima_params.npy")
sarima_params = np.load("sarima_params.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")
fc_m1 = np.load("fc_m1.npy")
fc_m2 = np.load("fc_m2.npy")
y_diff_arima = np.load("y_diff_arima.npy")
y_diff_sarima = np.load("y_diff_sarima.npy")
exog_full = np.load("exog_full.npy")
n_cut_sarima = int(np.load("n_cut_sarima.npy")[0])
info = np.load("model_info.npy")

aic_m1, bic_m1, aic_m2, bic_m2 = info[0], info[1], info[2], info[3]
negll_m1, negll_m2 = info[4], info[5]
n_params_m1, n_params_m2 = int(info[6]), int(info[7])
n_obs_m1, n_obs_m2 = int(info[8]), int(info[9])

df = pd.read_csv("retail_sales_mock_data.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df.set_index("Date", inplace=True)
n = len(df)
split = len(y_train)
dates = df.index
dates_train = dates[:split]
dates_test = dates[split:]

In [23]:
# Метрики прогноза на тестовой выборке

In [24]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

mse_m1 = mse(y_test, fc_m1)
rmse_m1 = rmse(y_test, fc_m1)
mae_m1 = mae(y_test, fc_m1)
mape_m1 = mape(y_test, fc_m1)
r2_m1 = r2(y_test, fc_m1)

mse_m2 = mse(y_test, fc_m2)
rmse_m2 = rmse(y_test, fc_m2)
mae_m2 = mae(y_test, fc_m2)
mape_m2 = mape(y_test, fc_m2)
r2_m2 = r2(y_test, fc_m2)

header = f"{'Метрика':>12}  {'M1: ARIMA(1,1,1)':>18}  {'M2: SARIMAX':>18}"
print(header)
metrics_table = [
    ("MSE", mse_m1, mse_m2),
    ("RMSE", rmse_m1, rmse_m2),
    ("MAE", mae_m1, mae_m2),
    ("MAPE, %", mape_m1, mape_m2),
    ("R²", r2_m1, r2_m2),]
for name, v1, v2 in metrics_table:
    better = "←" if v1 < v2 else ("→" if v2 < v1 else "=")
    if name == "R²":
        better = "←" if v1 > v2 else ("→" if v2 > v1 else "=")
    print(f"{name:>12}  {v1:>18.4f}  {v2:>18.4f}   {better}")

print("\nИнформационные критерии (на обучающей выборке):")
print(f"{'':>12}  {'M1: ARIMA(1,1,1)':>18}  {'M2: SARIMAX':>18}")
print("-" * 62)
for name, v1, v2 in [("AIC", aic_m1, aic_m2), ("BIC", bic_m1, bic_m2)]:
    better = "←" if v1 < v2 else "→"
    print(f"{name:>12}  {v1:>18.2f}  {v2:>18.2f}   {better}")

     Метрика    M1: ARIMA(1,1,1)         M2: SARIMAX
         MSE        7089753.7132        5988267.7198   →
        RMSE           2662.6591           2447.0937   →
         MAE           2257.2567           2113.2171   →
     MAPE, %             19.5222             17.0571   →
          R²             -0.3892             -0.1734   →

Информационные критерии (на обучающей выборке):
                M1: ARIMA(1,1,1)         M2: SARIMAX
--------------------------------------------------------------
         AIC              678.63              460.77   →
         BIC              685.08              468.08   →


In [25]:
# Вычисление остатков обеих моделей

In [26]:
def get_residuals_arima(params, y_diff, p, q, include_const=True):
    idx = 0
    mu = params[idx] if include_const else 0.0; idx += include_const
    phi = params[idx: idx+p]; idx += p
    theta = params[idx: idx+q]
    n_y = len(y_diff)
    res = np.zeros(n_y)
    for t in range(n_y):
        pred = mu
        for i, ph in enumerate(phi):
            if t-i-1 >= 0: pred += ph * y_diff[t-i-1]
        for j, th in enumerate(theta):
            if t-j-1 >= 0: pred += th * res[t-j-1]
        res[t] = y_diff[t] - pred
    return res

def get_residuals_sarima(params, y_diff, p, q, P, Q, s=12, include_const=True, exog=None):
    idx = 0
    mu = params[idx] if include_const else 0.0; idx += include_const
    phi = params[idx: idx+p]; idx += p
    theta = params[idx: idx+q]; idx += q
    PHI = params[idx: idx+P]; idx += P
    THETA = params[idx: idx+Q]; idx += Q
    n_exog = exog.shape[1] if exog is not None else 0
    beta = params[idx: idx+n_exog]
    n_y = len(y_diff)
    res = np.zeros(n_y)
    for t in range(n_y):
        pred = mu
        for i, ph in enumerate(phi):
            if t-i-1 >= 0: pred += ph * y_diff[t-i-1]
        for j, th in enumerate(theta):
            if t-j-1 >= 0: pred += th * res[t-j-1]
        for I, PH in enumerate(PHI):
            lag = (I+1)*s
            if t-lag >= 0: pred += PH * y_diff[t-lag]
        for J, TH in enumerate(THETA):
            lag = (J+1)*s
            if t-lag >= 0: pred += TH * res[t-lag]
        if exog is not None: pred += np.dot(beta, exog[t])
        res[t] = y_diff[t] - pred
    return res

resid_m1 = get_residuals_arima(arima_params, y_diff_arima, p=1, q=1)
exog_for_sarima = exog_full[n_cut_sarima:split]
resid_m2 = get_residuals_sarima(sarima_params, y_diff_sarima, p=1, q=0, P=1, Q=0, s=12, include_const=True, exog=exog_for_sarima)
print(f"\nОстатки M1: среднее={resid_m1.mean():.4f}, std={resid_m1.std():.4f}")
print(f"Остатки M2: среднее={resid_m2.mean():.4f}, std={resid_m2.std():.4f}")


Остатки M1: среднее=41.3485, std=2086.9894
Остатки M2: среднее=-102.2187, std=1910.2060


In [27]:
# Статистические тесты на остатках

In [28]:
# --- Нормальность ---
sw1, p_sw1 = stats.shapiro(resid_m1)
sw2, p_sw2 = stats.shapiro(resid_m2)

# --- Тест Льюнга–Бокса на автокоррелированность (вручную) ---
def ljung_box_test(resid, nlags=10):
    # Статистика Льюнга–Бокса Q и p-значение.
    # H0: остатки — белый шум (нет автокорреляции до лага h).
    n = len(resid)
    mean_r = resid.mean()
    var_r = np.sum((resid - mean_r)**2) / n
    acf_r = []
    for k in range(1, nlags+1):
        cov_k = np.sum((resid[k:]-mean_r)*(resid[:n-k]-mean_r)) / n
        acf_r.append(cov_k / var_r if var_r > 0 else 0)
    acf_r = np.array(acf_r)
    Q = n * (n+2) * np.sum(acf_r**2 / (n - np.arange(1, nlags+1)))
    p_val = 1 - stats.chi2.cdf(Q, df=nlags)
    return Q, p_val, acf_r

lb_Q1, lb_p1, acf_resid1 = ljung_box_test(resid_m1, nlags=10)
lb_Q2, lb_p2, acf_resid2 = ljung_box_test(resid_m2, nlags=10)

# --- Гомоскедастичность: тест Голдфелда–Квандта (разбивка пополам) ---
def goldfeld_quandt(resid):
   # Простейшая версия теста ГK: сравниваем дисперсии первой и второй половин.
    # H0: дисперсия однородна (гомоскедастична).
    # Статистика F ~ F(n/2-1, n/2-1).
    n = len(resid)
    m = n // 2
    r1 = resid[:m];    r2 = resid[m:]
    s1 = np.var(r1, ddof=1)
    s2 = np.var(r2, ddof=1)
    F = s2 / s1 if s1 > 0 else np.nan
    p = 2 * min(stats.f.cdf(F, m-1, n-m-1), 1 - stats.f.cdf(F, m-1, n-m-1))
    return F, p

gq_F1, gq_p1 = goldfeld_quandt(resid_m1)
gq_F2, gq_p2 = goldfeld_quandt(resid_m2)

print(f"\n{'Тест':<30}  {'M1: ARIMA':>12}  {'M2: SARIMAX':>12}")
print(f"{'Шапиро–Уилк W':<30}  {sw1:>12.4f}  {sw2:>12.4f}")
print(f"{'p-значение':<30}  {p_sw1:>12.4f}  {p_sw2:>12.4f}")
print(f"{'Нормальность (p>0.05)':<30}  " f"{'Да' if p_sw1>0.05 else 'Нет':>12}  {'Да' if p_sw2>0.05 else 'Нет':>12}")
print()
print(f"{'Льюнг–Бокс Q (лаги 1-10)':<30}  {lb_Q1:>12.4f}  {lb_Q2:>12.4f}")
print(f"{'p-значение':<30}  {lb_p1:>12.4f}  {lb_p2:>12.4f}")
print(f"{'Белый шум (p>0.05)':<30}  " f"{'Да' if lb_p1>0.05 else 'Нет':>12}  {'Да' if lb_p2>0.05 else 'Нет':>12}")
print()
print(f"{'Голдфелд–Квандт F':<30}  {gq_F1:>12.4f}  {gq_F2:>12.4f}")
print(f"{'p-значение':<30}  {gq_p1:>12.4f}  {gq_p2:>12.4f}")
print(f"{'Гомоскедастич. (p>0.05)':<30}  " f"{'Да' if gq_p1>0.05 else 'Нет':>12}  {'Да' if gq_p2>0.05 else 'Нет':>12}")


Тест                               M1: ARIMA   M2: SARIMAX
Шапиро–Уилк W                         0.9784        0.9732
p-значение                            0.6760        0.7262
Нормальность (p>0.05)                     Да            Да

Льюнг–Бокс Q (лаги 1-10)              5.3291       11.8277
p-значение                            0.8681        0.2968
Белый шум (p>0.05)                        Да            Да

Голдфелд–Квандт F                     0.9035        0.4037
p-значение                            0.8380        0.1438
Гомоскедастич. (p>0.05)                   Да            Да


In [29]:
# Визуализация

In [30]:
os.makedirs("/Users/maksimtimosuk/Desktop/evaluation", exist_ok=True)
os.chdir("/Users/maksimtimosuk/Desktop/evaluation")  # куда сохраняем PNG

In [31]:
def plot_residuals_panel(resid, model_name, acf_r, lb_Q, lb_p, sw_stat, sw_p, gq_F, gq_p, axes_row):
    #Строит диагностические графики остатков для одной модели
    n_r = len(resid)
    ci  = 1.96 / np.sqrt(n_r)

    # (a) Остатки во времени
    ax = axes_row[0]
    ax.plot(np.arange(n_r), resid, color="#2c7bb6", lw=1.2)
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(2*resid.std(), color="red", lw=0.8, linestyle="--")
    ax.axhline(-2*resid.std(), color="red", lw=0.8, linestyle="--")
    ax.set_title(f"{model_name} — Остатки во времени", fontsize=9)
    ax.set_xlabel("Наблюдение"); ax.grid(alpha=0.3)

    # (b) Гистограмма + KDE
    ax = axes_row[1]
    ax.hist(resid, bins=10, density=True, color="#4dac26",
            edgecolor="white", alpha=0.7, label="Гистограмма")
    xr = np.linspace(resid.min()-200, resid.max()+200, 200)
    kde = stats.gaussian_kde(resid)
    ax.plot(xr, kde(xr), color="#b8860b", lw=2, label="KDE")
    # нормальная кривая
    ax.plot(xr, stats.norm.pdf(xr, resid.mean(), resid.std()),
            color="red", lw=1.5, linestyle="--", label="Норм.")
    ax.set_title(f"Распределение остатков\nSW: W={sw_stat:.3f}, p={sw_p:.3f}",
                  fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

    # (c) Q–Q plot
    ax = axes_row[2]
    (osm, osr), (slope, intercept, r_qq) = stats.probplot(resid, dist="norm")
    ax.scatter(osm, osr, s=20, color="#7b3294", alpha=0.8)
    ax.plot(osm, slope*np.array(osm)+intercept, "r--", lw=1.5)
    ax.set_title(f"Q–Q Plot (r={r_qq:.3f})", fontsize=9)
    ax.grid(alpha=0.3)

    # (d) ACF остатков
    ax = axes_row[3]
    lags_r = np.arange(1, len(acf_r)+1)
    ax.bar(lags_r, acf_r, color="#d7191c", width=0.5, alpha=0.7)
    ax.axhline(ci, linestyle="--", color="black", lw=1)
    ax.axhline(-ci, linestyle="--", color="black", lw=1)
    ax.set_title(f"ACF остатков\nLjung–Box Q={lb_Q:.2f}, p={lb_p:.3f}",
                  fontsize=9)
    ax.set_xlabel("Лаг"); ax.grid(alpha=0.3)

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle("Диагностика остатков моделей", fontsize=13)

plot_residuals_panel(resid_m1, "M1: ARIMA(1,1,1)", acf_resid1, lb_Q1, lb_p1, sw1, p_sw1, gq_F1, gq_p1, axes[0])
plot_residuals_panel(resid_m2, "M2: SARIMAX(1,1,0)(1,1,0)[12]", acf_resid2, lb_Q2, lb_p2, sw2, p_sw2, gq_F2, gq_p2, axes[1])
plt.tight_layout()
plt.savefig("eval_residuals.png", dpi=150, bbox_inches="tight")
plt.close()

# ── График 2: сравнение прогнозов на тесте + информационные критерии
fig2, axes2 = plt.subplots(1, 2, figsize=(15, 6))
fig2.suptitle("Сравнение прогнозов на тестовой выборке", fontsize=13)

ax = axes2[0]
x_pos = np.arange(len(y_test))
ax.plot(x_pos, y_test, color="black", lw=2, marker="o", ms=5, label="Факт")
ax.plot(x_pos, fc_m1,  color="#d7191c", lw=2, linestyle="--", marker="s",
        ms=5, label=f"M1 ARIMA  (RMSE={rmse_m1:.0f})")
ax.plot(x_pos, fc_m2,  color="#1a9641", lw=2, linestyle="-.", marker="^",
        ms=5, label=f"M2 SARIMAX (RMSE={rmse_m2:.0f})")
ax.set_xticks(x_pos)
ax.set_xticklabels([d.strftime("%Y-%m") for d in dates_test],
                    rotation=45, fontsize=7)
ax.set_title("Прогнозы vs. факт (тест)")
ax.legend(fontsize=9)
ax.set_ylabel("SalesAmount")
ax.grid(alpha=0.3)

# Сводная таблица метрик в виде бар-чарта
ax = axes2[1]
metric_names = ["MSE/1000", "RMSE", "MAE", "MAPE,%", "R²×100"]
vals_m1 = [mse_m1/1000, rmse_m1, mae_m1, mape_m1, r2_m1*100]
vals_m2 = [mse_m2/1000, rmse_m2, mae_m2, mape_m2, r2_m2*100]
x = np.arange(len(metric_names))
width = 0.35
ax.bar(x - width/2, vals_m1, width, color="#d7191c", alpha=0.75, label="M1 ARIMA")
ax.bar(x + width/2, vals_m2, width, color="#1a9641", alpha=0.75, label="M2 SARIMAX")
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=9)
ax.set_title("Сравнение метрик (нормализовано для отображения)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("eval_metrics.png", dpi=150, bbox_inches="tight")
plt.close()

# ── График 3: скользящая дисперсия остатков (гомоскедастичность)
fig3, axes3 = plt.subplots(1, 2, figsize=(13, 5))
fig3.suptitle("Гомоскедастичность остатков (скользящая дисперсия)", fontsize=12)

for ax, resid, name, color in zip(axes3, [resid_m1, resid_m2], ["M1: ARIMA(1,1,1)", "M2: SARIMAX"], ["#d7191c", "#1a9641"]):
    win = 5
    rolling_var = [np.var(resid[max(0,i-win):i+1]) for i in range(len(resid))]
    ax.fill_between(np.arange(len(resid)), rolling_var, alpha=0.4, color=color)
    ax.plot(np.arange(len(resid)), rolling_var, color=color, lw=2)
    ax.set_title(f"{name}\nГолдфелд–Квандт p={gq_p1 if 'M1' in name else gq_p2:.3f}")
    ax.set_xlabel("Наблюдение")
    ax.set_ylabel("Скользящая дисперсия")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("eval_heteroscedasticity.png", dpi=150, bbox_inches="tight")
plt.close()

In [32]:
# Сводная таблица для отчёта

In [33]:
rows = [
    ("MSE", f"{mse_m1:,.1f}", f"{mse_m2:,.1f}"),
    ("RMSE", f"{rmse_m1:.1f}", f"{rmse_m2:.1f}"),
    ("MAE", f"{mae_m1:.1f}", f"{mae_m2:.1f}"),
    ("MAPE, %", f"{mape_m1:.2f}", f"{mape_m2:.2f}"),
    ("R²", f"{r2_m1:.4f}", f"{r2_m2:.4f}"),
    ("AIC", f"{aic_m1:.2f}", f"{aic_m2:.2f}"),
    ("BIC", f"{bic_m1:.2f}", f"{bic_m2:.2f}"),
    ("Нормальность", "Да" if p_sw1 > 0.05 else "Нет", "Да" if p_sw2 > 0.05 else "Нет"),
    ("Белый шум", "Да" if lb_p1 > 0.05 else "Нет", "Да" if lb_p2 > 0.05 else "Нет"),
    ("Гомоскедастич.", "Да" if gq_p1 > 0.05 else "Нет", "Да" if gq_p2 > 0.05 else "Нет"),
]

print(f"{'':20}  {'ARIMA(1,1,1)':>14}  {'SARIMAX':>12}")
print("-" * 50)
for name, v1, v2 in rows:
    print(f"  {name:<18}  {v1:>14}  {v2:>12}")

                        ARIMA(1,1,1)       SARIMAX
--------------------------------------------------
  MSE                    7,089,753.7   5,988,267.7
  RMSE                        2662.7        2447.1
  MAE                         2257.3        2113.2
  MAPE, %                      19.52         17.06
  R²                         -0.3892       -0.1734
  AIC                         678.63        460.77
  BIC                         685.08        468.08
  Нормальность                    Да            Да
  Белый шум                       Да            Да
  Гомоскедастич.                  Да            Да


In [34]:
# Аналитические выводы

1. По прогнозным метрикам (MSE, RMSE, R²) модель M2
   (SARIMAX) {'превосходит M1' if rmse_m2 < rmse_m1 else 'сопоставима с M1'} на тестовой выборке.
   M2 RMSE = {rmse_m2:.0f},  M1 RMSE = {rmse_m1:.0f}.

2. По информационным критериям (AIC, BIC), рассчитанным
   на обучающей выборке, {'M2 предпочтительнее' if aic_m2 < aic_m1 else 'M1 предпочтительнее'}:
   M1 AIC={aic_m1:.1f}/BIC={bic_m1:.1f}, M2 AIC={aic_m2:.1f}/BIC={bic_m2:.1f}.
   Более низкий AIC/BIC указывает на лучший баланс между
   точностью аппроксимации и сложностью модели.

3. Анализ остатков:
   • Нормальность: оба ряда остатков {'близки к нормальным' if p_sw1>0.05 and p_sw2>0.05 else 'показывают отклонение от нормальности'}
     (тест Шапиро–Уилка).
   • Автокоррелированность: {'остатки M1 автокоррелированы' if lb_p1<=0.05 else 'остатки M1 — белый шум'}
     {'/ остатки M2 — белый шум' if lb_p2>0.05 else '/ остатки M2 автокоррелированы'}
     (тест Льюнга–Бокса, лаги 1–10).
   • Гомоскедастичность: дисперсия остатков
     {'однородна для обеих моделей' if gq_p1>0.05 and gq_p2>0.05 else 'неоднородна, что нарушает базовые предположения OLS'}.

4. ВЫВОД: предпочтительной является модель M2 — SARIMAX,
   так как она явно учитывает сезонность (период 12 мес.)
   и внешние факторы (промо-акции, праздники), что даёт
   более точный прогноз и остатки, лучше соответствующие
   предположениям белого шума. При наличии достаточно
   длинного ряда (≥ 5 лет) рекомендуется добавить
   компонент Q=1 для улучшения краткосрочных предсказаний.